# Student Performance Prediction System using Machine Learning

**Project**: Student Performance Prediction System
**Objective**: Predict student academic performance (Pass/Fail and Grades) using historical student records, identify at-risk students early, and benchmark 5 Machine Learning algorithms.

---
### Table of Contents
1. [Project Overview & Setup](#1.-Project-Overview--Setup)
2. [Dataset Loading & Inspection](#2.-Dataset-Loading--Inspection)
3. [Data Preprocessing & Cleaning](#3.-Data-Preprocessing--Cleaning)
4. [Exploratory Data Analysis (EDA)](#4.-Exploratory-Data-Analysis-(EDA))
5. [Feature Selection & Train-Test Split](#5.-Feature-Selection--Train-Test-Split)
6. [Model Training (5 Algorithms)](#6.-Model-Training-(5-Algorithms))
7. [Model Evaluation & Comparison](#7.-Model-Evaluation--Comparison)
8. [Model Serialization with Joblib](#8.-Model-Serialization-with-Joblib)
9. [Single Student Prediction & At-Risk Identification](#9.-Single-Student-Prediction--At-Risk-Identification)


## 1. Project Overview & Setup
In traditional education systems, identifying academically weak or at-risk students before final examinations is difficult. This project automates student performance prediction using Machine Learning on key indicators:
- **Study Hours**: Daily study time (hours/day)
- **Attendance**: Percentage of class attendance (%)
- **Previous Marks**: Past examination score (%)
- **Assignments**: Average score across coursework assignments
- **Internal Marks**: Continuous assessment and internal test marks


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score

# Visual styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (8, 5)
print('All core ML packages imported successfully!')


## 2. Dataset Loading & Inspection
We load the student dataset (`student_data.csv`) and inspect its schema, shape, and distributions.

In [ ]:
# Load dataset
data_path = os.path.join('..', 'data', 'student_data.csv') if os.path.exists(os.path.join('..', 'data', 'student_data.csv')) else os.path.join('data', 'student_data.csv')
df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(10)


In [ ]:
print('--- Dataset Summary Info ---')
df.info()
print('\n--- Descriptive Statistics ---')
df.describe().round(2)


## 3. Data Preprocessing & Cleaning
Data cleaning steps:
1. Check and handle missing values.
2. Remove duplicate records.
3. Encode the target column (`Pass` -> 1, `Fail` -> 0).

In [ ]:
# Check missing values
print('Missing Values:\n', df.isnull().sum())

# Remove duplicates
init_len = len(df)
df = df.drop_duplicates()
print(f'Deduplication complete. Removed {init_len - len(df)} duplicate records.')

# Target Encoding: Pass=1, Fail=0
df['Target_Encoded'] = df['Final_Result'].apply(lambda x: 1 if str(x).strip().lower() == 'pass' else 0)
print('\nTarget Class Balance:')
print(df['Final_Result'].value_counts())


## 4. Exploratory Data Analysis (EDA)
Visualize relationships between academic inputs and student outcomes.

In [ ]:
# Correlation Heatmap
features = ['Study_Hours', 'Attendance', 'Previous_Marks', 'Assignments', 'Internal_Marks']
plt.figure(figsize=(8, 6))
sns.heatmap(df[features].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=1.5)
plt.title('Correlation Heatmap: Academic Determinants', fontsize=13, fontweight='bold')
plt.show()


In [ ]:
# Academic Factors vs Final Result
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.boxplot(data=df, x='Final_Result', y='Study_Hours', palette={'Pass': '#2ecc71', 'Fail': '#e74c3c'}, ax=axes[0])
axes[0].set_title('Study Hours vs Final Result', fontweight='bold')

sns.boxplot(data=df, x='Final_Result', y='Attendance', palette={'Pass': '#2ecc71', 'Fail': '#e74c3c'}, ax=axes[1])
axes[1].set_title('Attendance vs Final Result', fontweight='bold')

sns.boxplot(data=df, x='Final_Result', y='Previous_Marks', palette={'Pass': '#2ecc71', 'Fail': '#e74c3c'}, ax=axes[2])
axes[2].set_title('Previous Marks vs Final Result', fontweight='bold')
plt.tight_layout()
plt.show()


## 5. Feature Selection & Train-Test Split
We split the dataset into **80% Training Data** and **20% Testing Data** using stratified sampling to preserve the target class balance, and apply feature scaling.

In [ ]:
X = df[features].values
y = df['Target_Encoded'].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

print(f'Training Samples: {X_train.shape[0]}')
print(f'Testing Samples:  {X_test.shape[0]}')


## 6. Model Training (5 Algorithms)
We train and benchmark 5 standard algorithms:
1. **Logistic Regression**
2. **Decision Tree Classifier**
3. **Random Forest Classifier**
4. **Support Vector Machine (SVM)**
5. **Gaussian Naive Bayes**


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(C=1.0, max_iter=500, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_split=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    'Support Vector Machine (SVM)': CalibratedClassifierCV(SVC(C=1.0, kernel='rbf', random_state=42), ensemble=False),
    'Naive Bayes': GaussianNB(),
}

trained_models = {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    trained_models[name] = clf
    print(f'Trained: {name}')


## 7. Model Evaluation & Comparison
We compute **Accuracy**, **Precision**, **Recall**, **F1 Score**, and **ROC-AUC** on the unseen 20% test partition.

In [ ]:
evaluation_list = []
for name, clf in trained_models.items():
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1] if hasattr(clf, 'predict_proba') else y_pred
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)
    
    evaluation_list.append({
        'Model': name,
        'Accuracy (%)': round(acc * 100, 2),
        'Precision (%)': round(prec * 100, 2),
        'Recall (%)': round(rec * 100, 2),
        'F1 Score (%)': round(f1 * 100, 2),
        'ROC-AUC': round(auc, 4),
    })

comparison_df = pd.DataFrame(evaluation_list).sort_values(by=['F1 Score (%)', 'Accuracy (%)'], ascending=False).reset_index(drop=True)
print('=== Model Benchmark Comparison Table ===')
comparison_df


In [ ]:
# Confusion Matrix & Classification Report of Top Model
best_name = comparison_df.iloc[0]['Model']
best_clf = trained_models[best_name]
y_pred_best = best_clf.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted Fail', 'Predicted Pass'], yticklabels=['Actual Fail', 'Actual Pass'])
plt.title(f'Confusion Matrix: {best_name}', fontweight='bold')
plt.show()

print(f'\nClassification Report ({best_name}):\n')
print(classification_report(y_test, y_pred_best, target_names=['Fail', 'Pass']))


## 8. Model Serialization with Joblib
Serialize the best trained model and scaler to `models/best_student_model.joblib`.

In [ ]:
models_dir = os.path.join('..', 'models') if os.path.exists(os.path.join('..', 'models')) else 'models'
os.makedirs(models_dir, exist_ok=True)
bundle_file = os.path.join(models_dir, 'best_student_model.joblib')

joblib.dump({
    'model': best_clf,
    'scaler': scaler,
    'model_name': best_name,
    'features': features,
}, bundle_file)
print(f'Successfully serialized model bundle to: {bundle_file}')


## 9. Single Student Prediction & At-Risk Identification
Demonstrate inference on an at-risk student record to generate Pass/Fail classification, failure probability, and assigned risk level.

In [ ]:
# Example: Striving student with low study hours and low attendance
sample_features = np.array([[2.0, 56.0, 42.0, 45.0, 48.0]])  # Study_Hours, Attendance, Previous_Marks, Assignments, Internal_Marks
sample_scaled = scaler.transform(sample_features)

pred = best_clf.predict(sample_scaled)[0]
prob_pass = best_clf.predict_proba(sample_scaled)[0][1]
prob_fail = 1.0 - prob_pass

if prob_fail >= 0.50 or sample_features[0][1] < 65.0:
    risk_level = 'High Risk'
elif prob_fail >= 0.28 or sample_features[0][1] < 75.0:
    risk_level = 'Medium Risk'
else:
    risk_level = 'Low Risk'

print('=== Individual Student Assessment ===')
print(f"Predicted Outcome:    {'Pass' if pred == 1 else 'Fail'}")
print(f'Pass Probability:     {prob_pass * 100:.1f}%')
print(f'Failure Probability:  {prob_fail * 100:.1f}%')
print(f'Assigned Risk Tier:   {risk_level}')
if risk_level == 'High Risk':
    print('Action: Immediate faculty counseling and remedial classes recommended.')
